# 03 · Target vs. actual time series

For the success check, extracts the **commanded (target)** and **measured (actual)** waveforms per shot for the two signals that matter most:

| Signal | Target | Actual |
|---|---|---|
| Plasma current | `PTDATA/IPTIPP` (A) | `PTDATA/IP` (A) |
| Total NBI power | `PTDATA/BMSPINJ` (MW) | Σ of 8 beams' `PABS` (W) |

Both target signals were validated against known actual values on reference shots before being trusted (NBI matched to ~1–2 %, consistent with beam shine-through loss).

**Output:** one JSON file per shot.

In [ ]:
import h5py
import os
import json
import numpy as np

print("Imports done")

In [ ]:
data_dir = "/path/to/big_d3d_data/d3d_time_series_data"
all_files = os.listdir(data_dir)
all_files_sorted = sorted(all_files)

print("Total shot files found:", len(all_files_sorted))

In [ ]:
# target vs. actual extraction — IP and total NBI power.
# Both target signals were validated against known actual values on shots
# 144187 and 144229 before being trusted:
#   - PTDATA/IPTIPP (target IP) vs PTDATA/IP (actual IP)
#   - PTDATA/BMSPINJ (target total NBI power, in MW) vs summed D3D beam PABS
#     (actual absorbed power, in W)

NBI_BEAMS = ["NB15L", "NB15R", "NB21L", "NB21R", "NB30L", "NB30R", "NB33L", "NB33R"]
print("NBI_BEAMS defined:", NBI_BEAMS)

def _get_full_series(shot_number, f, tree, key, tmin=0, tmax=6000):
    try:
        if key not in f[str(shot_number)][tree]:
            return None, None
        diag = f[str(shot_number)][tree][key]
        data = diag["data"][:]
        time = diag["dim0"][:]
        mask = (time >= tmin) & (time <= tmax)
        return time[mask].tolist(), data[mask].tolist()
    except KeyError:
        return None, None

def get_ip_series(shot_number, f):
    """Actual and target IP, both in Amps."""
    a_t, a_d = _get_full_series(shot_number, f, "PTDATA", "IP")
    t_t, t_d = _get_full_series(shot_number, f, "PTDATA", "IPTIPP")
    return {"actual": {"time": a_t, "data": a_d}, "target": {"time": t_t, "data": t_d}}

def get_nbi_power_series(shot_number, f):
    """Actual (summed PABS, Watts) and target (BMSPINJ, MW) total NBI power."""
    ref_time, total_actual = None, None
    for beam in NBI_BEAMS:
        key = f"\\D3D::TOP.NB.{beam}:PABS"
        if key not in f[str(shot_number)]["D3D"]:
            continue
        diag = f[str(shot_number)]["D3D"][key]
        data, time = diag["data"][:], diag["dim0"][:]
        if ref_time is None:
            ref_time = time
            total_actual = np.zeros(len(time))
        if len(data) == len(total_actual):
            total_actual += data

    a_t, a_d = None, None
    if ref_time is not None:
        mask = (ref_time >= 0) & (ref_time <= 6000)
        a_t, a_d = ref_time[mask].tolist(), total_actual[mask].tolist()

    t_t, t_d = _get_full_series(shot_number, f, "PTDATA", "BMSPINJ")

    return {"actual": {"time": a_t, "data": a_d, "units": "W"},
            "target": {"time": t_t, "data": t_d, "units": "MW"}}

print("Target extraction functions defined")

In [ ]:
def extract_target_series(shot_number, data_dir):
    filename = os.path.join(data_dir, f"{shot_number}.h5")
    if not os.path.exists(filename):
        return None
    try:
        with h5py.File(filename, "r") as f:
            if str(shot_number) not in f:
                return None
            return {"shot_number": shot_number,
                    "ip": get_ip_series(shot_number, f),
                    "nbi_power": get_nbi_power_series(shot_number, f)}
    except Exception as e:
        print(f"Error on shot {shot_number}: {e}")
        return None

print("Master function defined")

In [ ]:
for test_shot in [144187, 144229]:
    result = extract_target_series(test_shot, data_dir)
    print(f"Shot {test_shot}:")
    for signal_name in ["ip", "nbi_power"]:
        for kind in ["actual", "target"]:
            data = result[signal_name][kind]["data"]
            n = len(data) if data else 0
            peak = max(data) if data else None
            print(f"  {signal_name} {kind}: {n} samples, max={peak}")
    print()

In [ ]:
output_dir = "/path/to/outputs/shot_targets_json"
failed_file = "/path/to/outputs/failed_target_shots.txt"
os.makedirs(output_dir, exist_ok=True)

total, success, failed = len(all_files_sorted), 0, 0

with open(failed_file, "w") as failfile:
    for i, filename in enumerate(all_files_sorted):
        if not filename.endswith(".h5"):
            continue
        name_part = filename.replace(".h5", "")
        if not name_part.isdigit():
            continue
        shot_number = int(name_part)
        if i % 200 == 0:
            print("Progress:", i, "/", total, "| succeeded:", success, "| failed:", failed)
        result = extract_target_series(shot_number, data_dir)
        if result is None:
            failed += 1
            failfile.write(str(shot_number) + "\n")
            continue
        with open(os.path.join(output_dir, f"{shot_number}.json"), "w") as out:
            json.dump(result, out)
        success += 1

print("Succeeded:", success, "| Failed:", failed)
print("Saved to:", output_dir)